In [ ]:
import geopandas as gpd
from shapely import geometry

from dep_tools.grids import PACIFIC_EPSG

In [ ]:
# Configure the resolution of the grid
RESOLUTION = 20
INPUT_BATHY = "/Users/alex/Data/bathymetry/TV.gpkg"
REGION_NAME = "Funafuti"

In [ ]:
# Load the regions file, and select our region
regions = gpd.read_file("postcards.geojson")

# Select the region we want
region = regions[regions['name'] == REGION_NAME]

region.explore()

In [ ]:
# Pick the datafile that matches the region in the below step
data = gpd.read_file(INPUT_BATHY, bbox=region.boundary)

# If a value is positive, make it negative
data['depth'] = data['depth'].apply(lambda x: x if x < 0 else -x)

# Filter out data that is very deep
data = data[data['depth'] > -100]

data = data.to_crs(PACIFIC_EPSG)

data

In [ ]:
# Get minX, minY, maxX, maxY
minX, minY, maxX, maxY = data.total_bounds

# Create a fishnet
x, y = (minX, minY)
geom_array = []

# Polygon Size
square_size = 20
while y <= maxY:
    while x <= maxX:
        geom = geometry.Polygon([(x,y), (x, y+square_size), (x+square_size, y+square_size), (x+square_size, y), (x, y)])
        geom_array.append(geom)
        x += square_size
    x = minX
    y += square_size

fishnet = gpd.GeoDataFrame(geom_array, columns=['geometry']).set_crs(PACIFIC_EPSG)

print(f"Created fishnet with {len(fishnet)} polygons")

In [ ]:
# Combine the fishnet with the bathymetry and get the average depth
joined = fishnet.sjoin(data, how="right", predicate="intersects")
print(f"Joined has {len(joined)} rows")

non_null = joined[joined['depth'].notna()]
print(f"Non-null has {len(non_null)} rows")

# Get the mean, median and stdev depth and the geometry
grouped = non_null.groupby('index_left')['depth'].agg(['mean', 'median', 'std'])
print(f"Grouped has {len(grouped)} rows")

# Join index_left to the fishnet
fishnet['index_left'] = fishnet.index
fishnet = fishnet.set_index('index_left')
joined_fishnet = fishnet.join(grouped, how='inner')

joined_fishnet["geometry"] = joined_fishnet["geometry"].centroid

# Rename mean to depth
joined_fishnet = joined_fishnet.rename(columns={"mean": "depth"})
print(f"Final fishnet has {len(joined_fishnet)} rows")

In [ ]:
joined_fishnet.explore(column="depth", cmap="viridis", legend=True, tooltip=True, name="Fishnet Depths")

In [ ]:
joined_fishnet.head(1000).explore(column="depth")

In [ ]:
joined_fishnet.to_file(f"data/{region['name'].values[0]}_{RESOLUTION}.gpkg", overwrite=True)

print(f"Wrote {len(joined_fishnet)} points")